# 第 2 節｜20 分鐘實作：交叉驗證與集成學習

使用同一批校園郵件，練習五折 CV、Grid Search、隨機森林與 Boosting。

In [ ]:
# Load Pandas for email data and comparison tables.
# 載入 Pandas，整理郵件與比較結果。
import pandas as pd

# Load Matplotlib for model comparison.
# 載入 Matplotlib，呈現模型比較。
import matplotlib.pyplot as plt

# Find the dataset in Colab or local folders.
# 使用 Path 找到 Colab 或本機的資料集。
from pathlib import Path

# Split data and perform cross-validation.
# 匯入資料切分、交叉驗證與網格搜尋工具。
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Train an SVM classifier.
# 匯入 SVM 分類模型。
from sklearn.svm import SVC

# Train a decision tree classifier.
# 匯入決策樹分類模型。
from sklearn.tree import DecisionTreeClassifier

# Train the bagging-inspired forest and boosting model.
# 匯入隨機森林與 AdaBoost 集成模型。
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# Evaluate all models with the same F1 score.
# 匯入 F1 分數，公平比較不同模型。
from sklearn.metrics import f1_score

## 1. 讀取相同的 500 封郵件

先把 Test 保留起來，只在 Train 上進行五折交叉驗證與超參數搜尋。

In [ ]:
# Find the CSV in Colab first.
# 優先從 Colab 的 /content 讀取資料。
file_path = Path("/content/02_校園郵件分類_500筆.csv")

# Use the local folder when not running on Colab.
# 如果不是在 Colab，就使用目前工作目錄。
if not file_path.exists():
    # Read from the current notebook folder.
    # 從目前 Notebook 工作目錄讀取。
    file_path = Path("02_校園郵件分類_500筆.csv")

# Load the 500 simulated school emails.
# 讀取 500 筆模擬校園電子郵件資料。
df = pd.read_csv(file_path)

In [ ]:
# Separate the email features from the correct labels.
# 將郵件特徵與正確答案分開。
X = df.drop(columns=["email_id", "is_spam"])

# Set aside 20 percent as untouched test data.
# 先保留 20% 當測試集，並維持垃圾郵件比例。
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, df["is_spam"], test_size=0.2, stratify=df["is_spam"], random_state=42)

In [ ]:
# Encode the training categories as binary columns.
# 將訓練資料的郵件類型轉成 One-Hot 欄位。
X_train = pd.get_dummies(X_train_raw, dtype=int)

# Match the test columns to the training feature columns.
# 測試資料沿用相同欄位，確保模型輸入一致。
X_test = pd.get_dummies(X_test_raw, dtype=int).reindex(columns=X_train.columns, fill_value=0)

## 2. 五折交叉驗證（5-Fold CV）

每一折都重新訓練模型；除了看平均，也看各折 F1 是否差很多。

In [ ]:
# Create an SVM for cross-validation.
# 建立要進行五折交叉驗證的 SVM。
svm = SVC(kernel="rbf")

# Evaluate five different training-validation splits.
# 使用五折交叉驗證計算五次 F1。
cv_scores = cross_val_score(svm, X_train, y_train, cv=5, scoring="f1")

# Display the five validation scores.
# 顯示五折驗證的 F1 分數。
print("五折 F1:", cv_scores.round(2))

# Display the average validation score.
# 顯示五折的平均 F1。
print("平均 F1:", round(cv_scores.mean(), 2))

## 3. Grid Search：挑選 SVM 的 C 與 gamma

測試 3×3 共 9 種設定，每種都使用五折 CV；**不使用測試集挑參數**。

In [ ]:
# Search a small grid of nine SVM configurations.
# 以 9 種 C／gamma 組合搜尋適合的參數。
grid = GridSearchCV(SVC(kernel="rbf"), {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1]}, cv=5, scoring="f1")

# Fit the search on training data only.
# 只用訓練資料進行搜尋與重新訓練。
grid.fit(X_train, y_train)

# Show the best hyperparameters in this search range.
# 顯示這次搜尋範圍內最佳的超參數。
print("最佳設定:", grid.best_params_)

# Show the best cross-validation F1.
# 顯示最佳設定的平均驗證 F1。
print("CV F1:", round(grid.best_score_, 2))

## 4. Ensemble Learning：Decision Tree、Random Forest、AdaBoost

隨機森林結合多棵樹；AdaBoost 依序關注較難預測的資料。

In [ ]:
# Build a shallow standalone decision tree.
# 建立單一決策樹，作為比較基準。
tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Build a forest of multiple decision trees.
# 建立結合多棵決策樹的隨機森林。
forest = RandomForestClassifier(n_estimators=60, max_depth=5, random_state=42)

# Build a simple sequential boosting classifier.
# 建立依序結合弱分類器的 AdaBoost。
boost = AdaBoostClassifier(n_estimators=40, random_state=42)

In [ ]:
# Train the standalone decision tree.
# 使用訓練資料訓練單棵決策樹。
tree.fit(X_train, y_train)

# Train the random forest.
# 使用相同資料訓練隨機森林。
forest.fit(X_train, y_train)

# Train the boosting classifier.
# 使用相同資料訓練 AdaBoost。
boost.fit(X_train, y_train);

## 5. 相同 Test、相同 F1：公平比較四種模型

這是最後的測試結果，**不要再反覆根據 Test 改超參數**。

In [ ]:
# Calculate one final test F1 for each model.
# 針對相同測試資料計算四種模型的 F1。
scores = [f1_score(y_test, tree.predict(X_test)), f1_score(y_test, forest.predict(X_test)), f1_score(y_test, boost.predict(X_test)), f1_score(y_test, grid.predict(X_test))]

# Plot the four test F1 scores for comparison.
# 以長條圖比較四個模型在測試集的 F1。
plt.bar(["Tree", "Forest", "AdaBoost", "SVM"], scores)

# Keep the F1 axis on a common zero-to-one scale.
# 將 F1 軸固定在 0 到 1，避免視覺誤導。
plt.ylim(0, 1)

# Label the evaluation metric.
# 標示這張圖比較的是測試集 F1。
plt.ylabel("Test F1")

# Display the model comparison chart.
# 顯示模型比較圖表。
plt.show()

## 6. 三封新郵件：用選好設定的 SVM 預測

Grid Search 已自動以最佳設定重新訓練模型，可以直接 `predict()`。

In [ ]:
# Create three new emails without correct-answer labels.
# 建立三筆沒有預先給答案的新郵件。
new_data = pd.DataFrame({
    "link_count": [0, 2, 5],
    "urgent_words": [0, 2, 4],
    "exclamation_count": [0, 2, 4],
    "sender_known": [1, 0, 0],
    "email_type": ["校務公告", "社團活動", "優惠資訊"]
})

# Reuse the exact feature columns learned during training.
# 新郵件必須和訓練資料使用相同的 One-Hot 欄位。
new_X = pd.get_dummies(new_data, dtype=int).reindex(columns=X_train.columns, fill_value=0)

# Use the selected SVM to classify the three new emails.
# 使用最佳設定的 SVM 預測三封新郵件。
new_data["預測（1=垃圾）"] = grid.predict(new_X)

# Show input values alongside their predicted labels.
# 顯示三封郵件的資料與預測結果。
new_data